In [ ]:
%pip install chronos-forecasting
%pip install ipywidgets
%pip install transformers accelerate


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import joblib
import pandas as pd
import numpy as np
import findspark
import pyspark as spark
from pyspark.sql import SparkSession
from chronos import Chronos2Pipeline
from pyspark.sql.types import StructType, StructField,FloatType,TimestampType,StringType,ArrayType
from pyspark.sql.functions import col,to_json,struct,from_json,explode

In [2]:
pipeline_pm10 = Chronos2Pipeline.from_pretrained("../Offline-Phase/my_chronos_pipeline_pm10")
pipeline_pm25 = Chronos2Pipeline.from_pretrained("../Offline-Phase/my_chronos_pipeline_pm25")

In [3]:
feature_scaler = joblib.load("../Offline-Phase/feature_scaler.pkl")
pm10_scaler_obj = joblib.load("../Offline-Phase/pm10_scaler.pkl")
pm25_scaler_obj = joblib.load("../Offline-Phase/pm25_scaler.pkl")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### This is a choice of which value does the model want to predict pm10 or pm25

In [6]:
choice = int(input("Enter 1 or 2 for the type of prediction (1:pm10 or 2:pm25): "))
choice

2

### Pandas Functions from the offline phase

In [7]:
def extract_time_features(df, timestamp_col='timestamp'):


    # 1. Cyclic Hour (24-hour cycle) //
    df['hour_sin'] = np.sin(2 * np.pi * df[timestamp_col].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df[timestamp_col].dt.hour / 24)

    # 2. Cyclic Month (12-month cycle)//
    df['month_sin'] = np.sin(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df[timestamp_col].dt.month - 1) / 12)

    # 3. Weekend Flag
    # Monday=0, Sunday=6. Weekends are 5 and 6.
    df['is_weekend'] = df[timestamp_col].dt.dayofweek.isin([5, 6]).astype(int)

    # 4. Heating Season Flag (Specific for Bitola/Macedonia)
    # Most wood burning occurs from November (11) to March (3)
    df['is_heating_season'] = df[timestamp_col].dt.month.isin([11, 12, 1, 2, 3]).astype(int)

    return df

In [8]:
def append_neighbors(df_hourly, neighbors_df, weather_cols, k_search=20, k_keep=3):
    # 1. Standardize the neighbor list
    # Ensure we only take the top K based on distance
    neighbors_topk = (
        neighbors_df.sort_values(["sensor_id", "distance_km"])
        .groupby("sensor_id")
        .head(k_search)
        .copy()
    )

    # Track original distance rank
    neighbors_topk['dist_rank'] = neighbors_topk.groupby("sensor_id").cumcount() + 1

    # 2. Merge with main data
    # We use 'neighbor_id' from the matrix to match 'sensorId' in the hourly data
    neighbor_values = neighbors_topk.merge(
        df_hourly[['sensorId', 'timestamp'] + weather_cols],
        left_on='neighbor_id',
        right_on='sensorId',
        how='inner'
    )

    # 3. Filter for availability
    # The 'sensor_id' here is the ORIGINAL sensor we are finding neighbors for
    available_topk = (
        neighbor_values.sort_values(['sensor_id', 'timestamp', 'dist_rank'])
        .groupby(['sensor_id', 'timestamp'])
        .head(k_keep)
        .copy()
    )

    # Create the 1, 2, 3 rank for the wide-format columns
    available_topk['final_rank'] = available_topk.groupby(['sensor_id', 'timestamp']).cumcount() + 1

    # 4. Pivot to wide format
    pivot_df = available_topk.pivot(
        index=['sensor_id', 'timestamp'],
        columns='final_rank',
        values=weather_cols
    )

    # Clean up column names: neighbor1_temp, neighbor2_temp, etc.
    if isinstance(pivot_df.columns, pd.MultiIndex):
        pivot_df.columns = [f"neighbor{rank}_{col}" for col, rank in pivot_df.columns]
    else:
        # Handle case with only one weather column
        pivot_df.columns = [f"neighbor{i}_{weather_cols[0]}" for i in pivot_df.columns]

    pivot_df = pivot_df.reset_index()

    # 5. Final Join back to original data
    df_result = df_hourly.merge(
        pivot_df,
        left_on=['sensorId', 'timestamp'],
        right_on=['sensor_id', 'timestamp'],
        how='left'
    ).drop(columns=['sensor_id'])

    return df_result

In [9]:
neighbourhood_matrix = pd.read_csv("../data/neighbors_data/bitola_sensor_distances.csv")
neighbourhood_matrix

,sensor_id,neighbor_id,distance_km
0,d241a044-0a06-40c2-9d90-c91fd0a95060,fec52a19-9148-4350-a1b4-ae0da05ee199,7.082000
1,fec52a19-9148-4350-a1b4-ae0da05ee199,d241a044-0a06-40c2-9d90-c91fd0a95060,7.082000
2,d241a044-0a06-40c2-9d90-c91fd0a95060,be427cee-4c3a-4aa2-a1ce-9795a74533be,8.838588
3,be427cee-4c3a-4aa2-a1ce-9795a74533be,d241a044-0a06-40c2-9d90-c91fd0a95060,8.838588
4,d241a044-0a06-40c2-9d90-c91fd0a95060,c3f3da9b-9fd3-4037-94d3-598d655e6be9,10.004966
...,...,...,...
457,7b316592-8036-41e2-b8dc-b06b6a9afd54,40f081a6-4095-43f7-bffb-64e2af8c026e,1.049671
458,40f081a6-4095-43f7-bffb-64e2af8c026e,692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,3.044465
459,692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,40f081a6-4095-43f7-bffb-64e2af8c026e,3.044465
460,7b316592-8036-41e2-b8dc-b06b6a9afd54,692c454c-a1ad-41fa-b3ca-aa1cb7d55d30,2.083640


In [10]:
def load_context(choice):
    if choice == 1:
        context_df = pd.read_csv("Context_pm10_bitola.csv")
    else:
        context_df = pd.read_csv("Context_pm25_bitola.csv")
    context_df['timestamp'] = pd.to_datetime(context_df['timestamp'])
    return context_df
    

In [11]:
def process_batch(batch_df,context_df):
    TARGET_COL = "pm10" if choice == 1 else "pm25"
    ID_COL = "sensorId"
    TIME_COL = "timestamp"
    print("Batch received!")
    print(batch_df.count())
    if batch_df.count() == 0:
        return None,None


    pdf = batch_df.toPandas()
    if pdf.empty:
        print("Empty batch after filtering — skipping")
        return None,None
    
    pdf["timestamp"] = pd.to_datetime(pdf["timestamp"],utc=True)
    pdf = extract_time_features(pdf)

    pdf = append_neighbors(
        pdf,
        neighbourhood_matrix,
        weather_cols=["humidity", "pressure","temperature", "wind_speed"]
    )
    numeric_features = [
    'humidity', 'pressure', 'temperature', 'wind_speed',
    'neighbor1_humidity', 'neighbor2_humidity', 'neighbor3_humidity',
    'neighbor1_pressure', 'neighbor2_pressure', 'neighbor3_pressure',
    'neighbor1_temperature', 'neighbor2_temperature', 'neighbor3_temperature',
    'neighbor1_wind_speed', 'neighbor2_wind_speed', 'neighbor3_wind_speed'
    ]
    pdf[numeric_features] = feature_scaler.transform(pdf[numeric_features])
    pdf = pdf.sort_values([ID_COL, TIME_COL])
    for col in numeric_features:
        pdf[col] = pdf[col].astype(context_df[col].dtype)
        
    context_df[numeric_features] = feature_scaler.transform(context_df[numeric_features])
    if TARGET_COL == "pm10":
        context_df['pm10'] = pm10_scaler_obj.transform(context_df[['pm10']])
        
        forecast_df = pipeline_pm10.predict_df(
            df=context_df,
            prediction_length=24,
            target=TARGET_COL,
            id_column=ID_COL,
            future_df=pdf,
            validate_inputs=False  
        )
    else:
        context_df['pm25'] = pm25_scaler_obj.transform(context_df[['pm25']])
        forecast_df = pipeline_pm25.predict_df(
            df=context_df,
            prediction_length=24,
            target=TARGET_COL,
            id_column=ID_COL,
            future_df=pdf,
            validate_inputs=False  
        )
   
    eval_df = pdf.merge(
    forecast_df[[ID_COL, TIME_COL, "predictions"]],
    on=[ID_COL, TIME_COL],
    how="left"
    )
    eval_df[numeric_features] = feature_scaler.inverse_transform(eval_df[numeric_features])
    if TARGET_COL == "pm10":
        eval_df["predictions"] = pm10_scaler_obj.inverse_transform(
            eval_df[["predictions"]]
        )
        eval_df = eval_df.rename(columns={"predictions": "pm10"})
        print(eval_df['pm10'].head(10))
    else:
        eval_df["predictions"] = pm25_scaler_obj.inverse_transform(
            eval_df[["predictions"]]
        )
        eval_df = eval_df.rename(columns={"predictions": "pm25"})
        print(eval_df['pm25'].head(10))
        
    
    return eval_df,eval_df

    

In [12]:
def write_to_kafka(df,choice):
    spark_df = spark.createDataFrame(df)

    kafka_df = spark_df.select(
        col("sensorId").cast("string").alias("key"),
        to_json(struct(*spark_df.columns)).alias("value")
    )

    if choice == 1:
        kafka_df.write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("topic", "FullPm10WeatherData") \
        .save()
    else:
        kafka_df.write \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("topic", "FullPm25WeatherData") \
        .save()

In [21]:
context_df = load_context(choice)

def foreach_batch(batch_df, epoch_id):
    global context_df
    result_df,new_context = process_batch(batch_df, context_df)
    if result_df is None and new_context is None:
        print("Skipping batch")
        return
    updated_context = pd.concat([context_df,new_context])
    context_df = updated_context.sort_values(["sensorId", "timestamp"]) \
                                .groupby("sensorId") \
                                .tail(512) \
                                .reset_index(drop=True)
    write_to_kafka(result_df,choice)
    print(f"Online Batch {epoch_id}: Context updated. Total records in memory: {len(context_df)}")

# Online Phase (Main Program)

In [14]:
findspark.init()

In [15]:
spark = SparkSession.builder \
    .appName("KafkaConsumerExample") \
    .config( "spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7")  \
    .getOrCreate()

:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/jovan/.ivy2/cache
The jars for the packages stored in: /Users/jovan/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-13e2a445-2bd2-4870-862e-e3ba366edae4;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 191ms :: artifacts dl 4ms
	:: 

In [16]:
df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "rawSensorWeatherData") \
    .load()

In [17]:
df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [18]:
schema = StructType([
    StructField("timestamp",TimestampType(),True),
    StructField("sensorId",StringType(),True),
    StructField("lat",FloatType(),True),
    StructField("lon",FloatType(),True),
    StructField("humidity",FloatType(),True),
    StructField("pressure",FloatType(),True),
    StructField("temperature",FloatType(),True),
    StructField("wind_speed",FloatType(),True)
])

In [19]:
parsed_df = df.select(
    from_json(col("value").cast("string"), ArrayType(schema)).alias("data")
).select(explode("data").alias("record")).select("record.*").drop("lat","lon")


In [20]:
parsed_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- humidity: float (nullable = true)
 |-- pressure: float (nullable = true)
 |-- temperature: float (nullable = true)
 |-- wind_speed: float (nullable = true)



In [22]:
query = parsed_df.writeStream \
    .foreachBatch(foreach_batch) \
    .start()

query.awaitTermination()

26/04/02 12:14:52 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/18/8m8jfl3d72v4p728z2p8fn8c0000gn/T/temporary-d71e8af7-0568-4a9a-a5b9-68fcb1b53caf. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/02 12:14:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/02 12:14:52 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


Batch received!
0
Skipping batch
Batch received!


264
0    21.139835
1    17.846394
2    15.850212
3    14.638903
4    14.960176
5    15.765451
6    19.941719
7    21.176559
8    18.585663
9    14.788246
Name: pm25, dtype: float32


26/04/02 12:15:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Online Batch 1: Context updated. Total records in memory: 5632
Batch received!
264


...

264
0    11.533330
1    11.495697
2    11.580168
3    11.541252
4    11.589746
5    11.573581
6    11.501036
7    11.361271
8    11.308741
9    11.263992
Name: pm25, dtype: float32
Online Batch 16: Context updated. Total records in memory: 5632
Batch received!
264


0    11.024397
1    11.002807
2    11.026005
3    10.966047
4    11.004919
5    10.974874
6    10.974220
7    10.972566
8    10.977617
9    10.955299
Name: pm25, dtype: float32
Online Batch 17: Context updated. Total records in memory: 5632
Batch received!
264
0    10.800779
1    10.909747
2    11.031406
3    11.163681
4    11.276192
5    11.322354
6    11.304745
7    11.204523
8    11.239835
9    11.235126
Name: pm25, dtype: float32
Online Batch 18: Context updated. Total records in memory: 5632
Batch received!
264
0    11.303970
1    11.304716
2    11.347490
3    11.340949
4    11.380512
5    11.383461
6    11.372142
7    11.391361
8    11.448532
9    11.487266
Name: pm25, dtype: float32
Online Batch 19: Context updated. Total records in memory: 5632
Batch received!
264
0    10.635862
1    10.701250
2    10.832359
3    10.804676
4    10.767801
5    10.688210
6    10.655651
7    10.694519
8    10.759008
9    10.788722
Name: pm25, dtype: float32
Online Batch 20: Context updated. Total 

0    9.509689
1    9.506925
2    9.566560
3    9.534555
4    9.599592
5    9.662724
6    9.629851
7    9.589659
8    9.547851
9    9.472947
Name: pm25, dtype: float32
Online Batch 25: Context updated. Total records in memory: 5632


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 